# 05 — .NET 11: Decimal32/64/128 e Complex<T>
Kernel **JupyterNet** eseguito sul runtime **.NET 11** (RC2 al momento della stesura): `scripts/run-notebooks.ps1` lo avvia con `DOTNET_ROLL_FORWARD=LatestMajor` e `DOTNET_ROLL_FORWARD_TO_PRERELEASE=1`. Roslyn scripting compila contro gli assembly del runtime in esecuzione, quindi i nuovi tipi sono visibili senza ricompilare JupyterNet. Le librerie Numbers sono net11.0 e generiche su questi stessi formati; questo notebook usa direttamente i tipi del runtime, senza caricarle.

IEEE decimal: 7/16/34 cifre significative, formati a 32/64/128 bit; NaN, infinito e signed zero. `System.Decimal` usa 128 bit di storage ma coefficiente 96 bit, 28–29 cifre, scala 0–28 e niente NaN/infinito: NON è Decimal128.
Fonti: [overview](https://learn.microsoft.com/en-us/dotnet/core/whats-new/dotnet-11/overview), [Decimal32](https://learn.microsoft.com/en-us/dotnet/api/system.numerics.decimal32?view=net-11.0), [Complex<T>](https://learn.microsoft.com/en-us/dotnet/api/system.numerics.complex-1?view=net-11.0).

In [ ]:
#load "../shared/Kernel.csx"
using System.Numerics;
using System.Globalization;
using System.Runtime.InteropServices;
Console.WriteLine(RuntimeInformation.FrameworkDescription);
if (Environment.Version.Major < 11) throw new InvalidOperationException("Serve il runtime .NET 11: eseguire con scripts/run-notebooks.ps1 o impostare DOTNET_ROLL_FORWARD.");

In [ ]:
var culture = CultureInfo.InvariantCulture;
var a32 = Decimal32.Parse("0.1",culture);
var b32 = Decimal32.Parse("0.2",culture);
Console.WriteLine(a32+b32 == Decimal32.Parse("0.3",culture));
Console.WriteLine(Decimal64.Parse("1",culture)/Decimal64.Parse("3",culture));
Console.WriteLine(Decimal128.Parse("1",culture)/Decimal128.Parse("3",culture));
Console.WriteLine(Decimal32.One/Decimal32.Zero);
Console.WriteLine(Decimal32.IsNaN(Decimal32.Zero/Decimal32.Zero));
Console.WriteLine(Decimal32.Sqrt(Decimal32.Parse("2",culture)));
Console.WriteLine(Decimal32.Sin(Decimal32.Pi));

In [ ]:
T Sum11<T>(T[] values) where T : INumberBase<T>
{
    var total = T.Zero;
    foreach (var value in values) total += value;
    return total;
}
Console.WriteLine(Sum11(new[] {a32,b32}));
var zf = new Complex<float>(3f,4f);
Console.WriteLine(zf.GetMagnitude());
var zd = new Complex<Decimal64>(Decimal64.Parse("3",culture),Decimal64.Parse("4",culture));
Console.WriteLine(zd.GetMagnitude());
Console.WriteLine(Sum11(new[] {zd,zd}));

In [ ]:
var source = "12.5;resto";
if (Decimal32.TryParsePartial(source,NumberStyles.Float,culture,out var value,out var consumed))
    Console.WriteLine($"valore={value}; consumati={consumed}; resto={source.Substring(consumed)}");

## Esperimenti e limiti
Parse di 12345678 con Decimal32, Decimal64 e Decimal128: distinguere cifre significative da posti decimali. Provare `BitIncrement(One)`, `Quantize` e conversioni checked. I nuovi formati espongono funzioni trascendenti e IFloatingPointIeee754; `decimal` no. Complex<T> richiede IFloatingPointIeee754<T> e IMinMaxValue<T>: Complex<decimal> non è valido.
## Piano benchmark .NET 11 (non ancora implementato)
Ripetere lo stesso kernel Sum11 con array Decimal32/64/128 pre-parsati in GlobalSetup e confrontare con decimal/double; misurare separatamente Parse, Sqrt e Complex<float>/Complex<Decimal64>. Non includere parsing nel benchmark di somma. Nessun risultato prestazionale sui nuovi formati è stato raccolto: non presumere accelerazione hardware o equivalenza di precisione.